# RandomForest + FGM Dual-Stream Consistency Gate Pipeline

This notebook mirrors the LR and NN FGM consistency-gate workflow for a RandomForest classifier.

Important implementation note:

- FGM needs input gradients.
- A RandomForest does not provide those gradients.
- This notebook trains a small PyTorch surrogate only to generate FGM adversarial samples.
- The actual clean model, adversarially trained guardian model, evaluations, and consistency gate are all RandomForest-based.

Pipeline steps:

1. Train a clean RandomForest on clean training data
2. Train a small PyTorch surrogate and generate FGM adversarial samples
3. Evaluate the clean RandomForest on clean, adversarial, and combined test data
4. Train a guardian RandomForest on clean plus FGM samples
5. Evaluate the guardian RandomForest
6. Run the dual-stream consistency gate


In [5]:
# If needed, install once:
# !pip install torch adversarial-robustness-toolbox scikit-learn pandas numpy joblib

import warnings
warnings.filterwarnings("ignore")

import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import PyTorchClassifier

try:
    from IPython.display import display
except ImportError:
    display = print


In [ ]:
# -----------------------------
# Configuration
# -----------------------------
DATA_PATH = Path(r"..\..\CSVs\dataset.csv")

LABEL_COL = "anomaly"

# Drop non-feature columns if needed
DROP_COLS = {LABEL_COL, "segment", "train", "sampling", "channel"}

TEST_SIZE = 0.2
SEED = 42

# FGM settings
FGM_EPS = 0.10

# For sklearn models, adversarial training is approximated by adding
# FGM samples generated from the surrogate model back into the training set.
ADV_RATIO = 0.55

# Surrogate model settings used only for generating FGM samples.
SURROGATE_HIDDEN = 64
SURROGATE_LR = 1e-3
SURROGATE_BATCH_SIZE = 128
SURROGATE_EPOCHS = 20

# Save paths
SAVE_MODELS = True
ARTIFACT_DIR = Path(r"artifacts")
RESULTS_DIR = Path(r"results")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# RandomForest settings
BEST_PARAMS_PATH = Path(r"..\..\MachineLearning\RandomForest\best_params.csv")

DEFAULT_RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "bootstrap": True,
    "class_weight": None,
    "n_jobs": -1,
    "random_state": SEED,
}


In [7]:
from sklearn.ensemble import RandomForestClassifier

def coerce_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        text = value.strip()
        if text.lower() in {"none", "nan", ""}:
            return None
        if text.lower() in {"true", "false"}:
            return text.lower() == "true"
        try:
            return ast.literal_eval(text)
        except Exception:
            return text
    return value


def load_best_params(path: Path, defaults: dict):
    params = defaults.copy()

    if path.exists():
        best_df = pd.read_csv(path)
        row = best_df.iloc[0].to_dict()
        for key, value in row.items():
            if key in params:
                params[key] = coerce_value(value)
        print(f"Loaded RandomForest params from {path}")
    else:
        print(f"Best params file not found at {path}. Using DEFAULT_RF_PARAMS.")

    params["random_state"] = SEED
    params["n_jobs"] = -1

    return params


RF_PARAMS = load_best_params(BEST_PARAMS_PATH, DEFAULT_RF_PARAMS)
print("RF_PARAMS:", RF_PARAMS)


def make_rf_model():
    return RandomForestClassifier(**RF_PARAMS)


Best params file not found at ..\MachineLearning\RandomForest\best_params.csv. Using DEFAULT_RF_PARAMS.
RF_PARAMS: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'n_jobs': -1, 'random_state': 42}


In [8]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_df = df[[c for c in df.columns if c not in DROP_COLS]].copy()

    non_numeric_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric feature columns found in {csv_path}: {non_numeric_cols}. "
            "Add them to DROP_COLS or encode them before training."
        )

    feature_cols = feature_df.columns.tolist()
    X = feature_df.to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), scaler, feature_cols


if not DATA_PATH.exists():
    raise ValueError(f"Set DATA_PATH first. Current value does not exist: {DATA_PATH}")

X_train, X_test, y_train, y_test, scaler, feature_cols = load_and_prepare(str(DATA_PATH))


Loaded: ..\..\CSVs\dataset.csv
Rows=2123, Features=18, Label dist=[1689  434]
Train=(1698, 18), Test=(425, 18)


In [9]:
class SurrogateMLP(nn.Module):
    def __init__(self, d_in: int, hidden: int = SURROGATE_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x):
        return self.net(x)


def make_surrogate_art_classifier(d_in: int):
    model = SurrogateMLP(d_in=d_in)
    loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=SURROGATE_LR)

    return PyTorchClassifier(
        model=model,
        loss=loss,
        optimizer=optimizer,
        input_shape=(d_in,),
        nb_classes=2,
        clip_values=(0.0, 1.0),
    )


def predict_labels(model, X: np.ndarray):
    return model.predict(X).astype(int)


def predict_proba(model, X: np.ndarray):
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X)
    else:
        preds = model.predict(X)
        probs = np.zeros((len(preds), 2), dtype=np.float32)
        probs[np.arange(len(preds)), preds.astype(int)] = 1.0
    return probs.astype(np.float32)


def eval_sklearn_classifier(model, X: np.ndarray, y_true: np.ndarray, name: str):
    y_pred = predict_labels(model, X)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }, y_pred


def make_adv_training_set(X_clean, y_clean, X_adv, y_adv, adv_ratio: float = ADV_RATIO):
    n_adv = max(1, int(round(len(X_clean) * adv_ratio)))
    n_adv = min(n_adv, len(X_adv))

    rng = np.random.default_rng(SEED)
    adv_idx = rng.choice(len(X_adv), size=n_adv, replace=False)

    X_mix = np.vstack([X_clean, X_adv[adv_idx]]).astype(np.float32)
    y_mix = np.concatenate([y_clean, y_adv[adv_idx]]).astype(np.int64)

    shuffle_idx = rng.permutation(len(X_mix))
    return X_mix[shuffle_idx], y_mix[shuffle_idx]


In [10]:
# Step 1: train clean RandomForest model
print("Training clean RandomForest with:", RF_PARAMS)

clean_model = make_rf_model()
clean_model.fit(X_train, y_train)

clean_on_clean, y_pred_clean = eval_sklearn_classifier(
    clean_model,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

if SAVE_MODELS:
    joblib.dump(clean_model, ARTIFACT_DIR / "randomforest_clean_model.joblib")
    joblib.dump(scaler, ARTIFACT_DIR / "randomforest_scaler.joblib")


Training clean RandomForest with: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'n_jobs': -1, 'random_state': 42}

[clean_model_on_clean_test] acc=0.9529 f1=0.8750
confusion matrix:
[[335   3]
 [ 17  70]]
              precision    recall  f1-score   support

           0     0.9517    0.9911    0.9710       338
           1     0.9589    0.8046    0.8750        87

    accuracy                         0.9529       425
   macro avg     0.9553    0.8979    0.9230       425
weighted avg     0.9532    0.9529    0.9514       425



In [11]:
# Step 2: train surrogate and generate FGM adversarial samples
print("Training PyTorch surrogate used only for FGM generation with:", {
    "hidden": SURROGATE_HIDDEN,
    "learning_rate": SURROGATE_LR,
    "batch_size": SURROGATE_BATCH_SIZE,
    "epochs": SURROGATE_EPOCHS,
    "fgm_eps": FGM_EPS,
})

surrogate_art = make_surrogate_art_classifier(d_in=X_train.shape[1])
surrogate_art.fit(
    X_train,
    y_train,
    batch_size=SURROGATE_BATCH_SIZE,
    nb_epochs=SURROGATE_EPOCHS,
)

fgm = FastGradientMethod(estimator=surrogate_art, eps=FGM_EPS)

X_train_adv = fgm.generate(x=X_train).astype(np.float32)
X_test_adv = fgm.generate(x=X_test).astype(np.float32)

print("Adversarial data generated:")
print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# Keep adversarial labels aligned with original ground-truth labels.
y_train_adv = y_train.copy()
y_test_adv = y_test.copy()

X_test_combined = np.vstack([X_test, X_test_adv]).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv]).astype(np.int64)


Training PyTorch surrogate used only for FGM generation with: {'hidden': 64, 'learning_rate': 0.001, 'batch_size': 128, 'epochs': 20, 'fgm_eps': 0.1}
Adversarial data generated:
X_train_adv: (1698, 18)
X_test_adv: (425, 18)


In [12]:
# Evaluate clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_sklearn_classifier(
    clean_model,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_sklearn_classifier(
    clean_model,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)



[clean_model_on_adv_test] acc=0.1859 f1=0.2939
confusion matrix:
[[  7 331]
 [ 15  72]]
              precision    recall  f1-score   support

           0     0.3182    0.0207    0.0389       338
           1     0.1787    0.8276    0.2939        87

    accuracy                         0.1859       425
   macro avg     0.2484    0.4241    0.1664       425
weighted avg     0.2896    0.1859    0.0911       425


[clean_model_on_combined_test] acc=0.5694 f1=0.4369
confusion matrix:
[[342 334]
 [ 32 142]]
              precision    recall  f1-score   support

           0     0.9144    0.5059    0.6514       676
           1     0.2983    0.8161    0.4369       174

    accuracy                         0.5694       850
   macro avg     0.6064    0.6610    0.5442       850
weighted avg     0.7883    0.5694    0.6075       850



In [13]:
# Step 3: adversarial training approximation for RandomForest
X_train_mixed, y_train_mixed = make_adv_training_set(
    X_train,
    y_train,
    X_train_adv,
    y_train_adv,
    adv_ratio=ADV_RATIO,
)

print("Training guardian RandomForest with clean + FGM samples:")
print("X_train_mixed:", X_train_mixed.shape)
print("y_train_mixed:", y_train_mixed.shape)

adv_model = make_rf_model()
adv_model.fit(X_train_mixed, y_train_mixed)

if SAVE_MODELS:
    joblib.dump(adv_model, ARTIFACT_DIR / "randomforest_adversarial_trained_model.joblib")


Training guardian RandomForest with clean + FGM samples:
X_train_mixed: (2632, 18)
y_train_mixed: (2632,)


In [14]:
# Step 4: evaluate adversarially trained guardian model
adv_trained_on_adv, y_pred_adv = eval_sklearn_classifier(
    adv_model,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

adv_trained_on_clean, y_pred_clean_adv_model = eval_sklearn_classifier(
    adv_model,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_sklearn_classifier(
    adv_model,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)



[adv_trained_model_on_adv_test] acc=0.9435 f1=0.8537
confusion matrix:
[[331   7]
 [ 17  70]]
              precision    recall  f1-score   support

           0     0.9511    0.9793    0.9650       338
           1     0.9091    0.8046    0.8537        87

    accuracy                         0.9435       425
   macro avg     0.9301    0.8919    0.9093       425
weighted avg     0.9425    0.9435    0.9422       425


[adv_trained_model_on_clean_test] acc=0.9506 f1=0.8696
confusion matrix:
[[334   4]
 [ 17  70]]
              precision    recall  f1-score   support

           0     0.9516    0.9882    0.9695       338
           1     0.9459    0.8046    0.8696        87

    accuracy                         0.9506       425
   macro avg     0.9488    0.8964    0.9195       425
weighted avg     0.9504    0.9506    0.9491       425


[adv_trained_model_on_combined_test] acc=0.9471 f1=0.8615
confusion matrix:
[[665  11]
 [ 34 140]]
              precision    recall  f1-score   support


In [15]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_adv,
    adv_trained_on_clean,
    adv_trained_on_combined,
])

display(summary_df)


,model_eval,acc,f1
0,clean_model_on_clean_test,0.952941,0.875000
1,clean_model_on_adv_test,0.185882,0.293878
2,clean_model_on_combined_test,0.569412,0.436923
3,adv_trained_model_on_adv_test,0.943529,0.853659
4,adv_trained_model_on_clean_test,0.950588,0.869565
5,adv_trained_model_on_combined_test,0.947059,0.861538


In [16]:
# Save metrics
summary_path = RESULTS_DIR / "randomforest_fgm_pipeline_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")


Saved: results\randomforest_fgm_pipeline_summary.csv


## Dual-Stream Consistency Gate

This section mirrors the dual-stream detector structure from the LR and NN FGM notebooks.

### Gate logic

- **Nominal model** = clean model
- **Guardian model** = adversarially trained model
- Flag likely attacks using:
  1. prediction disagreement
  2. high-confidence disagreement
  3. nominal predicts benign while guardian predicts anomaly by a large enough margin

The gate is evaluated both as:

- a **final prediction system** for clean, adversarial, and combined inputs
- an **attack detector** using **FPR**, **TPR**, and **F1**


In [17]:
CONFIDENCE_THRESHOLD = 0.60
DISAGREEMENT_THRESHOLD = 0.55

def predict_with_confidence(model, X: np.ndarray):
    probs = predict_proba(model, X)
    preds = np.argmax(probs, axis=1)
    return preds, probs


class DualStreamDetector:
    def __init__(self, nominal_model, guardian_model):
        self.nominal_model = nominal_model
        self.guardian_model = guardian_model

    def detect_attacks(
        self,
        X: np.ndarray,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        disagreement_threshold: float = DISAGREEMENT_THRESHOLD,
    ):
        preds_nominal, probs_nominal = predict_with_confidence(self.nominal_model, X)
        preds_guardian, probs_guardian = predict_with_confidence(self.guardian_model, X)

        n_samples = len(X)
        flags = np.zeros(n_samples, dtype=int)
        details = []

        for i in range(n_samples):
            yA = int(preds_nominal[i])
            yB = int(preds_guardian[i])
            pA = probs_nominal[i]
            pB = probs_guardian[i]

            conf_nominal = float(pA[yA])
            conf_guardian = float(pB[yB])

            detected = False
            reasons = []

            if yA != yB:
                detected = True
                reasons.append("disagreement")

            if yA == 0 and yB == 1:
                prob_diff = float(pB[1] - pA[1])
                if conf_nominal >= confidence_threshold and conf_guardian >= confidence_threshold:
                    if prob_diff >= disagreement_threshold:
                        detected = True
                        reasons.append("high_confidence_disagreement")
            else:
                prob_diff = float(pB[1] - pA[1])

            flags[i] = int(detected)
            details.append({
                "nominal_pred": yA,
                "guardian_pred": yB,
                "nominal_conf": conf_nominal,
                "guardian_conf": conf_guardian,
                "nominal_anom_prob": float(pA[1]),
                "guardian_anom_prob": float(pB[1]),
                "prob_diff_anomaly": prob_diff,
                "detected": bool(detected),
                "reason": ",".join(reasons) if reasons else "none",
            })

        final_preds = np.where(flags == 1, preds_guardian, preds_nominal)
        details_df = pd.DataFrame(details)
        return final_preds, flags, details_df


def evaluate_dual_stream_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }


detector = DualStreamDetector(
    nominal_model=clean_model,
    guardian_model=adv_model,
)

gated_clean_pred, flags_clean, gate_clean_df = detector.detect_attacks(X_test)
gated_adv_pred, flags_adv, gate_adv_df = detector.detect_attacks(X_test_adv)
gated_combined_pred, flags_combined, gate_combined_df = detector.detect_attacks(X_test_combined)

gated_clean_metrics = evaluate_dual_stream_predictions(
    y_test, gated_clean_pred, "dual_stream_final_predictions_on_clean_test"
)
gated_adv_metrics = evaluate_dual_stream_predictions(
    y_test_adv, gated_adv_pred, "dual_stream_final_predictions_on_adv_test"
)
gated_combined_metrics = evaluate_dual_stream_predictions(
    y_test_combined, gated_combined_pred, "dual_stream_final_predictions_on_combined_test"
)

dual_stream_prediction_summary_df = pd.DataFrame([
    gated_clean_metrics,
    gated_adv_metrics,
    gated_combined_metrics,
])

print("\nDual-stream final prediction summary:")
display(dual_stream_prediction_summary_df)

y_attack_true = np.concatenate([
    np.zeros(len(flags_clean), dtype=int),
    np.ones(len(flags_adv), dtype=int),
])
y_attack_pred = np.concatenate([flags_clean, flags_adv])

dual_stream_detection_results = pd.DataFrame([{
    "attack": "FGM",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "disagreement_threshold": DISAGREEMENT_THRESHOLD,
    "FPR": float(flags_clean.mean()),
    "TPR": float(flags_adv.mean()),
    "F1": float(f1_score(y_attack_true, y_attack_pred, zero_division=0)),
    "clean_model_acc_on_adv": float(clean_on_adv["acc"]),
    "guardian_model_acc_on_clean": float(adv_trained_on_clean["acc"]),
    "guardian_model_acc_on_adv": float(adv_trained_on_adv["acc"]),
}])

print("\nDual-stream attack-detection summary:")
display(dual_stream_detection_results)

print("\nGate reason counts on clean test:")
display(gate_clean_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))

print("\nGate reason counts on adversarial test:")
display(gate_adv_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))



[dual_stream_final_predictions_on_clean_test] acc=0.9506 f1=0.8696
confusion matrix:
[[334   4]
 [ 17  70]]
              precision    recall  f1-score   support

           0     0.9516    0.9882    0.9695       338
           1     0.9459    0.8046    0.8696        87

    accuracy                         0.9506       425
   macro avg     0.9488    0.8964    0.9195       425
weighted avg     0.9504    0.9506    0.9491       425


[dual_stream_final_predictions_on_adv_test] acc=0.9435 f1=0.8537
confusion matrix:
[[331   7]
 [ 17  70]]
              precision    recall  f1-score   support

           0     0.9511    0.9793    0.9650       338
           1     0.9091    0.8046    0.8537        87

    accuracy                         0.9435       425
   macro avg     0.9301    0.8919    0.9093       425
weighted avg     0.9425    0.9435    0.9422       425


[dual_stream_final_predictions_on_combined_test] acc=0.9471 f1=0.8615
confusion matrix:
[[665  11]
 [ 34 140]]
              prec

,model_eval,acc,f1
0,dual_stream_final_predictions_on_clean_test,0.950588,0.869565
1,dual_stream_final_predictions_on_adv_test,0.943529,0.853659
2,dual_stream_final_predictions_on_combined_test,0.947059,0.861538



Dual-stream attack-detection summary:


,attack,confidence_threshold,disagreement_threshold,FPR,TPR,F1,clean_model_acc_on_adv,guardian_model_acc_on_clean,guardian_model_acc_on_adv
0,FGM,0.6,0.55,0.007059,0.837647,0.908163,0.185882,0.950588,0.943529



Gate reason counts on clean test:


,reason,count
0,none,422
1,disagreement,3



Gate reason counts on adversarial test:


,reason,count
0,disagreement,356
1,none,69


In [18]:
# Save dual-stream outputs
dual_stream_prediction_summary_path = RESULTS_DIR / "randomforest_dual_stream_prediction_summary.csv"
dual_stream_detection_path = RESULTS_DIR / "randomforest_dual_stream_detection_results.csv"

dual_stream_prediction_summary_df.to_csv(dual_stream_prediction_summary_path, index=False)
dual_stream_detection_results.to_csv(dual_stream_detection_path, index=False)

gate_clean_df.to_csv(RESULTS_DIR / "randomforest_dual_stream_gate_clean_details.csv", index=False)
gate_adv_df.to_csv(RESULTS_DIR / "randomforest_dual_stream_gate_adv_details.csv", index=False)
gate_combined_df.to_csv(RESULTS_DIR / "randomforest_dual_stream_gate_combined_details.csv", index=False)

print(f"Saved: {dual_stream_prediction_summary_path}")
print(f"Saved: {dual_stream_detection_path}")


Saved: results\randomforest_dual_stream_prediction_summary.csv
Saved: results\randomforest_dual_stream_detection_results.csv
